# GOAL: Build linear regression model

In [1]:
#import statements

#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error
#from statsmodels.tsa.statespace.sarimax import SARIMAX
import random
from sklearn.linear_model import LinearRegression


#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")


In [ ]:
# #no hyperparameters --> 1D array is enough, but will do 2D to make it generalizable
# all_errors = -np.ones((1,len(prerun.train_dates)))

# linReg = LinearRegression()
# i_date = 0
# system_recorded_max = prerun.data['energy'].max()
# for date in prerun.train_dates['date']:
#     #print(type(date))
#     data = prerun.data_until_ho_day(date)
#     data_train = data.iloc[:-2]
#     X_train = data_train.drop(columns = ['time','energy'])
#     y_train = data_train['energy']
#     data_ho = data.iloc[[-1]]
#     X_ho = data_ho.drop(columns = ['time','energy'])
#     y_ho = data_ho['energy']
#     linReg.fit(X_train, y_train)

#     y_pred = linReg.predict(X_ho)
#     #make sure value between 0 and highest observed max
#     y_pred = np.clip(y_pred, 0, system_recorded_max)
#     #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
#     darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
#     y_pred = y_pred*np.array(darkness_mask)
#     error = PostRun.custom_error(y_ho, y_pred)
#     all_errors[0,i_date] = error
#     i_date = i_date+1

# #calculate mean of each row
# hypper_means = np.mean(all_errors, axis=1)
# print(f"System {system_id} Linear Regression all error stats:")
# print(f"mean = {hypper_means}, std = {np.std(all_errors, axis = 1)}, min = {np.min(all_errors,axis=1)}, max = {np.max(all_errors, axis=1)}, median = {np.median(all_errors,axis=1)}")

# #plot
# plt.boxplot(all_errors[0], vert=False)  # horizontal → like a number line
# plt.xlabel("Errors")
# plt.title(f"System {system_id} Errors")
# plt.show()

# nonzero_errors = all_errors[0]
# nonzero_errors = nonzero_errors[nonzero_errors!=0]
# print(f"System {system_id} Linear Regression nonzero error stats:")
# print(f"mean = {np.mean(nonzero_errors)}, std = {np.std(nonzero_errors)}, min = {np.min(nonzero_errors)}, max = {np.max(nonzero_errors)}, median = {np.median(nonzero_errors)}")
# plt.boxplot(nonzero_errors, vert=False)  # horizontal → like a number line
# plt.xlabel("Errors")
# plt.title(f"System {system_id} Nonzero Errors")
# plt.show()


: 

: 

: 

In [2]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None),(1283,'inverter'),(1283,'meter')]
#system_reader_pairs = [(1283,'inverter'),(1283,'meter')]


for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(daily_lags = 2, remove_daily_lags_nans=True, include_last_year = True, remove_last_year_nans=True, include_day_of_year_cyclic=True, highest_fourier_term_hour = 3)

    prerun.add_weather_features_only()

    prerun.amended_data = prerun.amended_data.dropna()

    prerun.good_end_days_naive(1)
    prerun.tts_of_data_using_end_days()

    
    all_errors = []

    linReg = LinearRegression()
    system_recorded_max = prerun.data['energy'].max()
    for date in prerun.train_dates['date']:
        #print(type(date))
        data = prerun.data_until_ho_day(date)
        data_train = data.iloc[:-2]
        X_train = data_train.drop(columns = ['time','energy'])
        y_train = data_train['energy']
        data_ho = data.iloc[[-1]]
        X_ho = data_ho.drop(columns = ['time','energy'])
        y_ho = data_ho['energy']
        linReg.fit(X_train, y_train)

        y_pred = linReg.predict(X_ho)
        #make sure value between 0 and highest observed max
        y_pred = np.clip(y_pred, 0, system_recorded_max)
        #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
        y_pred = y_pred*np.array(darkness_mask)
        error = PostRun.custom_error(y_ho, y_pred)
        all_errors.append(error)

    #save errors
    errors = pd.DataFrame(all_errors, columns = ['error'])
    errors.to_csv(f'linreg_errors/{system_id}_{reader_type}_linreg_errors.csv', index=False)



look at error distributions

In [ ]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.003889790919288431, median: 0.00032833177732965, min: 0.0, max: 0.16890587553856, std: 0.009021006773062603

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.006385485653922355, median: 0.0024650655329941, min: 0.0, max: 0.1355389677625591, std: 0.010303435529115104

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.014225985586992276, median: 0.0012414392905465, min: 0.0, max: 0.7823899676964471, std: 0.04791271415016249

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.18190079969657835, median: 0.0558787705381852, min: 0.0, max: 2.3829944131305574, std: 0.26460121168134

System 51, None, LinReg with streak = 1
recorded system max: 7.2368749999999995
  Hyperparameters: error
     mean: 

#### linreg vs sarimax with best hyperparam. ALL data.

System 4: unknown (linreg has better mean, but worse max)

System 10: linreg

System 33: unknown (sarimax has better max; linreg has better everything else)

System 50: linreg (by a lot)

System 51: linreg (by a lot)

System 1283, inverter: unknown (linreg has WAY better mean, but WAY worse max)

System 1283, meter: linreg (by a lot)



note that linreg (and naive baseline) should both get better the later you go. So I'll run the above with only the second half of linreg, then redo the comparisons.

In [10]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.003022308810107891, median: 0.0002610685252368, min: 0.0, max: 0.0635133318922372, std: 0.006765800226416441

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.004965432432950153, median: 0.00160610776436125, min: 0.0, max: 0.0875471433155998, std: 0.008895054463369823

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.013339119449206014, median: 0.0012052242729839, min: 0.0, max: 0.7823899676964471, std: 0.04905521980893034

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.1647753628877309, median: 0.05742601218453745, min: 0.0, max: 1.13408731823097, std: 0.21309227363782648

System 51, None, LinReg with streak = 1
recorded system max: 7.2368749999999995
  Hyperparameters: error
     mea

#### linreg vs sarimax with best hyperparam. Only second half of linreg.

System 4: linreg (new!)

System 10: linreg

System 33: unknown (sarimax has better max; linreg has better everything else. Same max as before.)

System 50: linreg (by a lot)

System 51: linreg (by a lot)

System 1283, inverter: linreg (new!)

System 1283, meter: linreg (by a lot)


2nd half linreg is better than 2nd half naive across the board.



Now look at errors of linreg with more daily/hourly fourier terms (3)

In [3]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.004096970776078545, median: 0.0007096429119410001, min: 0.0, max: 0.16890587553856, std: 0.008853849940752171

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.006301510954218518, median: 0.00263045335448635, min: 0.0, max: 0.1454695489572813, std: 0.010567874465099207

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.016617770403714746, median: 0.0026770265134601, min: 0.0, max: 0.7337518504746614, std: 0.04732412135493175

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.128213172317279, median: 0.0322100252946336, min: 0.0, max: 2.533137496487409, std: 0.2224414143295617

System 51, None, LinReg with streak = 1
recorded system max: 7.2368749999999995
  Hyperparameters: error
     mean

and with only the second half of the outputs!

In [4]:
#compare hyperparameters
#System 4
print('System 4, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/4_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/10_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/33_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/50_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/51_None_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_inverter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, LinReg with streak = 1')
errors = pd.read_csv('linreg_errors/1283_meter_linreg_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, LinReg with streak = 1
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.0031964386355627025, median: 0.0005639525726395, min: 0.0, max: 0.0588801505658444, std: 0.006509889078949542

System 10, None, LinReg with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.005196297760606679, median: 0.0022341090173509997, min: 0.0, max: 0.085967027864706, std: 0.008881361858075093

System 33, None, LinReg with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.01582020292250979, median: 0.0026429643491706, min: 0.0, max: 0.7337518504746614, std: 0.04842562989314636

System 50, None, LinReg with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.10932576380916277, median: 0.0264273075240546, min: 0.0, max: 1.0841377046529337, std: 0.18041003769196237

System 51, None, LinReg with streak = 1
recorded system max: 7.2368749999999995
  Hyperparameters: error
     

## Compute on test sets!!!

Since LinReg won the race

In [ ]:
system_reader_pairs = [(10,None), (50,None), (51,None)]
#system_reader_pairs = [(1283,'inverter'),(1283,'meter')]


for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(daily_lags = 2, remove_daily_lags_nans=True, include_last_year = True, remove_last_year_nans=True, include_day_of_year_cyclic=True, highest_fourier_term_hour = 3)

    prerun.add_weather_features_only()

    prerun.amended_data = prerun.amended_data.dropna()

    prerun.good_end_days_naive(1)
    prerun.tts_of_data_using_end_days()

    
    all_errors = []

    linReg = LinearRegression()
    system_recorded_max = prerun.data['energy'].max()
    for date in prerun.test_dates['date']:
        #print(type(date))
        data = prerun.data_until_ho_day(date)
        data_train = data.iloc[:-2]
        X_train = data_train.drop(columns = ['time','energy'])
        y_train = data_train['energy']
        data_ho = data.iloc[[-1]]
        X_ho = data_ho.drop(columns = ['time','energy'])
        y_ho = data_ho['energy']
        linReg.fit(X_train, y_train)

        y_pred = linReg.predict(X_ho)
        #make sure value between 0 and highest observed max
        y_pred = np.clip(y_pred, 0, system_recorded_max)
        #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
        y_pred = y_pred*np.array(darkness_mask)
        error = PostRun.custom_error(y_ho, y_pred)
        all_errors.append(error)

    #save errors
    errors = pd.DataFrame(all_errors, columns = ['error'])
    errors.to_csv(f'final_linreg_errors/{system_id}_{reader_type}_linreg_errors.csv', index=False)

